# Problem 3 (programming parts). A β-Weighted Variational Autoencoder

**Day 2 · Problem 3 · Parts 3.5–3.8 (25 of the problem's 50 points).**
The shared setup and the theory Parts 3.1–3.4 are in `theory/p03.md`; read its shared setup first.
All reductions are the shared setup's: **sum over features** in the reconstruction term, **sum over latents** in the KL, **mean over the batch**, and $\beta$ multiplies the KL term only.
Throughout, $\beta=2$ unless a part states otherwise.

**Allowed tools.** Python's standard library, NumPy, and PyTorch only, on a CPU. No web access, downloads, external datasets, model hubs, pretrained weights, or checkpoints you did not create in the same run.
`torch.distributions` is not needed and may not be used for the KL or for sampling.

**Data.** `data/p03_vae.json` holds 8-feature rows: `"train"` (256 rows) and `"heldout"` (64 rows).
The cell below loads them as `float32` tensors `x_train` `(256, 8)` and `x_heldout` `(64, 8)`.
Held-out rows never enter `backward()` and are used for no choice.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()

vae_rows = json.loads((DATA_DIR / "p03_vae.json").read_text(encoding="utf-8"))
x_train = torch.tensor(vae_rows["train"], dtype=torch.float32)
x_heldout = torch.tensor(vae_rows["heldout"], dtype=torch.float32)
BETA = 2.0
print(x_train.shape, x_heldout.shape)


## Part 3.5 (5 points)

**Type:** programming · **Answer form:** code · Coding allowed.

Write `kl_to_standard_normal(mu, logvar)` returning the per-example KL of the shared setup, shape `(B,)` for inputs `(B, k)`, and `reparameterize(mu, logvar, eps)` returning $z=\mu+\exp(\frac12\,\texttt{logvar})\odot\varepsilon$ (the noise `eps` is an argument; the function draws no randomness).

**Probe** (all `float64`): `mu = [[1.0, 0.0], [0.5, -0.5]]`, `logvar = [[0.0, ln 4], [ln 0.25, 0.0]]`, `eps = [[1.0, -1.0], [2.0, 0.0]]`, with `mu` and `logvar` created with `requires_grad=True`.

**Required prints and assertions.** Print `kl` and `z`.
`kl[0]` equals your Part 3.1 answer (tolerance `1e-12`);
`z` equals `[[2, -2], [1.5, -0.5]]` (tolerance `1e-12`);
after `z.sum().backward()`, `mu.grad` is all ones and `logvar.grad` equals `0.5 * exp(0.5 * logvar) * eps` (tolerance `1e-12`).
**Answer:** print `float(kl[1])` to six decimal places.

In [ ]:
def kl_to_standard_normal(mu, logvar):
    raise NotImplementedError


def reparameterize(mu, logvar, eps):
    raise NotImplementedError


# Part 3.5 probe, prints, and assertions.


## Part 3.6 (5 points)

**Type:** programming · **Answer form:** code · Coding allowed.

Write `beta_vae_loss(x, x_hat, mu, logvar, beta)` returning a tuple `(total, recon_mean, kl_mean)` of 0-dimensional tensors, where `recon_mean` is the batch mean of the per-example reconstruction term, `kl_mean` the batch mean of the per-example KL (use `kl_to_standard_normal`), and `total = recon_mean + beta * kl_mean` (the batch β-weighted objective).

**Probe.** The two-example batch of Part 3.2 (`float64`), with `beta = 2.0`.

**Required prints and assertions.** Print all three numbers.
`total` equals your Part 3.2 answer (tolerance `1e-12`);
`total == recon_mean + 2 * kl_mean` (tolerance `1e-12`);
with `beta = 1.0` on the same tensors the total equals `recon_mean + kl_mean`.
**Answer:** print `float(total)` to six decimal places.

In [ ]:
def beta_vae_loss(x, x_hat, mu, logvar, beta):
    raise NotImplementedError


# Part 3.6 probe, prints, and assertions.


## Part 3.7 (10 points)

**Type:** programming · **Answer form:** code · Coding allowed.

Write `class TinyVAE(nn.Module)` with constructor `TinyVAE()` creating, **in this order**: `self.hidden = nn.Linear(8, 16)`, `self.mu_head = nn.Linear(16, 2)`, `self.logvar_head = nn.Linear(16, 2)`, `self.dec1 = nn.Linear(2, 16)`, `self.dec2 = nn.Linear(16, 8)`, and methods

* `encode(x)` returning `(mu, logvar)` with `h = F.relu(self.hidden(x))`, `mu = self.mu_head(h)`, `logvar = self.logvar_head(h)`;
* `decode(z)` returning `self.dec2(F.relu(self.dec1(z)))`;
* `forward(x, eps)` returning `(x_hat, mu, logvar)`: encode, `reparameterize`, decode.

**Training protocol** (a function `train_vae(beta)` that returns the trained model, the per-update log, and the held-out numbers is recommended, because Part 3.8 reruns it):

1. `torch.manual_seed(SEED)` immediately before `model = TinyVAE()`; then `optimizer = torch.optim.Adam(model.parameters(), lr=0.01)`.
2. `eps_gen = torch.Generator().manual_seed(SEED)` for training noise; `eval_gen = torch.Generator().manual_seed(SEED + 1)` and, once, `eps_heldout = torch.randn(64, 2, generator=eval_gen)`.
3. Held-out objective before training: under `torch.no_grad()`, `beta_vae_loss(x_heldout, *model(x_heldout, eps_heldout), beta)`'s `total`.
4. Exactly **400** full-batch updates on `x_train`; each update, in this order: `eps = torch.randn(256, 2, generator=eps_gen)`, `optimizer.zero_grad(set_to_none=True)`, forward, `beta_vae_loss(..., beta)`, `total.backward()`, `optimizer.step()`; log `total`, `recon_mean`, and `kl_mean` as floats.
5. Held-out objective after training, with the same `eps_heldout`.

Run it with `beta = BETA` (= 2).

**Required prints and assertions.**
every logged update satisfies `total == recon + 2 * kl` (tolerance `1e-5`);
the mean of the last 20 logged totals is below the first logged total;
held-out after `<= 0.8 *` held-out before;
a gradient-flow probe on the trained model: with `e = torch.randn(256, 2, generator=torch.Generator().manual_seed(SEED + 2))`, the **reconstruction term alone** (`recon_mean` from `beta_vae_loss`) backpropagated gives a `model.mu_head.weight.grad` that is not `None` and not all zero.
State the held-out number as a "β-weighted objective (β = 2), nats per example, constant dropped", not as an ELBO.
**Answer:** print the held-out objective after training to three decimal places.

In [ ]:
class TinyVAE(nn.Module):
    def __init__(self):
        super().__init__()
        raise NotImplementedError

    def encode(self, x):
        raise NotImplementedError

    def decode(self, z):
        raise NotImplementedError

    def forward(self, x, eps):
        raise NotImplementedError


# Part 3.7 training, prints, and assertions.


## Part 3.8 (5 points)

**Type:** programming · **Answer form:** code · Coding allowed.

For a trained model, the **per-latent held-out KL** is the length-2 vector whose coordinate $j$ is the average over the 64 held-out rows of $\frac12(\mu_j^2+e^{\ell_j}-\ell_j-1)$, with `(mu, logvar) = model.encode(x_heldout)` (no noise).
A latent coordinate is **active** when its per-latent held-out KL exceeds `0.05` nats.

(a) For the β = 2 model of Part 3.7, print the per-latent held-out KL and the number of active coordinates.

(b) Rerun the complete Part 3.7 protocol from scratch with `beta = 8.0` (same seeds and generators) and print the same two things.

(c) In at most two sentences, name the failure the β = 8 run shows and one log quantity from Part 3.7's protocol that reveals it.

**Required assertions.** Both per-latent vectors have shape `(2,)` and nonnegative entries; the two active counts are integers.
**Answer:** print the two active counts as `active(beta=2), active(beta=8)`, for example `5, 7`.

In [ ]:
# Part 3.8 diagnostics, rerun at beta = 8, prints, and assertions.


## Your answer to Part 3.8(c) (at most two sentences)

*Replace this line.*